# SF1: Total allele lengths at disease-associated loci

In [ ]:
import pandas as pd
totalLen=pd.read_csv('statsTables/DiscoveryCohort_alleleLengthStats.TR_Explorer_1.0.1.txt.gz',sep='\t')

# Define TRIDs
pathogenic=pd.read_csv('assets/pathogenicTRIDs_TRExplorer_v1.0.1_STRchiveFiltered_noVNTRs_withAnno.txt',sep='\t')

# select pathogenic TRs
totalLen=pathogenic.loc[:,['TRID','geneName']].merge(totalLen,how='inner',on='TRID')
totalLen.loc[:,['geneName','TRID','N','0thPercentile','1stPercentile','5thPercentile','10thPercentile','15thPercentile','20thPercentile','25thPercentile','30thPercentile','35thPercentile',
                '40thPercentile','45thPercentile','50thPercentile','55thPercentile','60thPercentile','65thPercentile','70thPercentile','75thPercentile','80thPercentile',
                '85thPercentile','90thPercentile','95thPercentile','99thPercentile','99.9thPercentile','100thPercentile']].to_csv('statsTables/SupplementaryFile1.txt',sep='\t',index=False)

# SF2: LPS lengths at disease-associated loci

In [ ]:
import pandas as pd
lps=pd.read_csv('statsTables/DiscoveryCohort_lpsPerMotifStats.TR_Explorer_1.0.1.txt.gz',sep='\t')

# Define TRIDs
pathogenic=pd.read_csv('assets/pathogenicTRIDs_TRExplorer_v1.0.1_STRchiveFiltered_noVNTRs_withAnno.txt',sep='\t')

# select pathogenic TRs
lps=pathogenic.loc[:,['TRID','geneName']].merge(lps,how='inner',on='TRID')
lps.loc[:,['geneName','TRID','LPS_Motif','N_motif','0thPercentile','1stPercentile','5thPercentile','10thPercentile','15thPercentile','20thPercentile','25thPercentile','30thPercentile','35thPercentile',
                '40thPercentile','45thPercentile','50thPercentile','55thPercentile','60thPercentile','65thPercentile','70thPercentile','75thPercentile','80thPercentile',
                '85thPercentile','90thPercentile','95thPercentile','99thPercentile','99.9thPercentile','100thPercentile']].to_csv('statsTables/SupplementaryFile2.txt',sep='\t',index=False)

# SF3: Pathogenicity classification performance

In [ ]:
# Report sensitivity and specificity of PLVI for the disease-associated STRs. Do the same for each genomic feature and RExPRT. 
import os
import pickle

import numpy as np
import pandas as pd
import scipy.stats


BASE_DIR = os.path.dirname('.')
STATS_DIR = os.path.join(BASE_DIR, 'statsTables')
ASSETS_DIR = os.path.join(BASE_DIR, 'assets')
ENRICH_DIR = os.path.join(ASSETS_DIR, 'pathoEnrichment')
PERCENTILE_THRESHOLDS = [95, 97, 99, 99.9]

def roc_auc_score(y_true, y_score):
    """AUROC via the Mann-Whitney U identity: U / (n_pos * n_neg).

    Ties contribute 0.5. Matches sklearn.metrics.roc_auc_score for
    binary labels.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    if pos.size == 0 or neg.size == 0:
        return float('nan')
    u, _ = scipy.stats.mannwhitneyu(pos, neg, alternative='greater')
    return u / (pos.size * neg.size)


def confusion_counts(y_true, y_pred):
    """Return (tn, fp, fn, tp) for binary 0/1 arrays."""
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tn, fp, fn, tp


def f1_from_counts(tp, fp, fn):
    denom = 2 * tp + fp + fn
    return (2 * tp / denom) if denom > 0 else 0.0


def evaluate_score(scores, labels, direction, fixed_thresholds=None):
    """Return AUROC plus per-threshold sensitivity / specificity / F1.

    scores           -- 1-D numeric array of the score being evaluated
    labels           -- 1-D binary array (1 = pathogenic, 0 = other)
    direction        -- 'higher' if larger scores predict pathogenic,
                        'lower'  if smaller scores predict pathogenic
    fixed_thresholds -- optional dict mapping tag -> raw threshold value.
                        Scores at/beyond the threshold (in the pathogenic
                        direction) are called positive.
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    mask = ~np.isnan(scores)
    s = scores[mask]
    y = labels[mask]

    signed = s if direction == 'higher' else -s
    auroc = roc_auc_score(y, signed)

    metrics = {
        'n_total': int(len(y)),
        'n_pathogenic': int(y.sum()),
        'direction': direction,
        'auroc': auroc,
    }

    def _record(tag, thresh):
        if direction == 'higher':
            preds = (s >= thresh).astype(int)
        else:
            preds = (s <= thresh).astype(int)

        tn, fp, fn, tp = confusion_counts(y, preds)
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = f1_from_counts(tp, fp, fn)

        metrics[f'threshold_{tag}'] = thresh
        metrics[f'sensitivity_{tag}'] = sens
        metrics[f'specificity_{tag}'] = spec
        metrics[f'f1_{tag}'] = f1
        metrics[f'tp_{tag}'] = int(tp)
        metrics[f'fp_{tag}'] = int(fp)
        metrics[f'fn_{tag}'] = int(fn)
        metrics[f'tn_{tag}'] = int(tn)

    for p in PERCENTILE_THRESHOLDS:
        if direction == 'higher':
            thresh = np.percentile(s, p)
        else:
            thresh = np.percentile(s, 100 - p)
        _record(f'p{p}'.replace('.', '_'), thresh)

    if fixed_thresholds:
        for tag, thresh in fixed_thresholds.items():
            _record(tag, thresh)

    return metrics


def complement(seq):
    comp = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(comp[b] for b in seq)


def reverse_complement(seq):
    return complement(seq[::-1])


def simplest_motif(motif):
    i = (motif + motif).find(motif, 1, -1)
    if i != -1:
        motif = motif[:i]
    motifs = [(motif[i:] + motif[:i]).upper() for i in range(len(motif))]
    motif_rc = reverse_complement(motif.upper())
    i = (motif_rc + motif_rc).find(motif_rc, 1, -1)
    if i != -1:
        motif_rc = motif_rc[:i]
    motifs += [motif_rc[i:] + motif_rc[:i] for i in range(len(motif_rc))]
    return min(motifs)


def load_pathogenic_ids():
    pathogenic = pd.read_csv(
        os.path.join(ASSETS_DIR, 'pathogenicTRIDs_TRExplorer_v1.0.1_STRchiveFiltered_noVNTRs_withAnno.txt'),
        sep='\t',
    )
    poly_alanine_names = ['ARX', 'NIPA1', 'HOXA13', 'PHOX2B', 'SOX3',
                          'HOXD13', 'ZIC2', 'RUNX2', 'PABPN1', 'FOXL2']
    single_shift_names = ['MIR7-2', 'COMP']
    excluded = set(poly_alanine_names) | set(single_shift_names)
    other_pathogenic = pathogenic.loc[~pathogenic['geneName'].isin(excluded), 'TRID'].values
    return other_pathogenic


def run_allele_length_block(other_pathogenic_ids, results):
    total_len = pd.read_csv(
        os.path.join(STATS_DIR, 'DiscoveryCohort_alleleLengthStats.TR_Explorer_1.0.1.txt.gz'),
        sep='\t',
    )
    total_len['patho'] = total_len['TRID'].isin(other_pathogenic_ids).astype(int)

    columns = ['std', '50thPercentile', '99thPercentile', '99.9thPercentile',
               '100thPercentile', 'mean', 'mad', 'mode',
               'uniqueLengths', 'uniqueAlleleSeqs']
    for column in columns:
        sub = total_len[[column, 'patho']].dropna(subset=[column])
        metrics = evaluate_score(sub[column].values, sub['patho'].values, direction='higher')
        metrics['condition'] = f'Allele Length {column}'
        results.append(metrics)

    total_len[['chr', 'start', 'end', 'refMotif']] = (
        total_len['TRID'].str.split(',', expand=True).iloc[:, 0].str.split('-', expand=True)
    )
    total_len['motifLen'] = total_len['refMotif'].str.len()
    total_len['percentile_stdev_ref_norm'] = (
        total_len.groupby('motifLen', observed=True)['std'].rank(pct=True)
    )
    sub = total_len[['percentile_stdev_ref_norm', 'patho']].dropna(subset=['percentile_stdev_ref_norm'])
    metrics = evaluate_score(sub['percentile_stdev_ref_norm'].values, sub['patho'].values, direction='higher',fixed_thresholds={'binary95': 0.95, 'binary97':0.97,'binary99':0.99,'binary999':0.999})
    metrics['condition'] = 'Allele Length (Ref Motif Normalized)'
    results.append(metrics)

    return total_len


def run_lps_block(other_pathogenic_ids, results):
    lps = pd.read_csv(
        os.path.join(STATS_DIR, 'DiscoveryCohort_lpsPerMotifStats.TR_Explorer_1.0.1.txt.gz'),
        sep='\t'
    )
    lps = (lps.sort_values(by=['TRID', 'N_motif'], ascending=False)
              .drop_duplicates(subset='TRID', keep='first')
              .reset_index(drop=True))
    lps['simpleMotif'] = lps['LPS_Motif'].apply(simplest_motif)
    lps['motifLen'] = lps['simpleMotif'].str.len()
    lps['patho'] = lps['TRID'].isin(other_pathogenic_ids).astype(int)

    for column in ['std', 'mean', 'mad', 'mode']:
        sub = lps[[column, 'patho']].dropna(subset=[column])
        metrics = evaluate_score(sub[column].values, sub['patho'].values, direction='higher')
        metrics['condition'] = f'Per Motif LPS Length {column}'
        results.append(metrics)

    lps['percentile_stdev_motif_norm'] = (
        lps.groupby('motifLen', observed=True)['std'].rank(pct=True)
    )
    sub = lps[['percentile_stdev_motif_norm', 'patho']].dropna(subset=['percentile_stdev_motif_norm'])
    metrics = evaluate_score(sub['percentile_stdev_motif_norm'].values, sub['patho'].values, direction='higher',
                                 fixed_thresholds={'binary95': 0.95, 'binary97':0.97,'binary99':0.99,'binary999':0.999})
    metrics['condition'] = 'Per Motif LPS Length (LPS Motif Normalized)'
    results.append(metrics)

    # add on LPS per locus stats
    locus = pd.read_csv(
        os.path.join(STATS_DIR, 'DiscoveryCohort_lpsPerLocusStats.TR_Explorer_1.0.1.txt.gz'),
        sep='\t'
    )
    locus=locus.merge(lps.loc[:,['TRID','motifLen','patho']],how='inner',on='TRID')
    locus['PLVI']=locus.groupby('motifLen', observed=True)['std'].rank(pct=True)
    for column in ['std','PLVI']:
        sub = locus[[column, 'patho']].dropna(subset=[column])
        if column=='PLVI':
            metrics = evaluate_score(sub[column].values, sub['patho'].values, direction='higher',fixed_thresholds={'binary95': 0.95, 'binary97':0.97,'binary99':0.99,'binary999':0.999})
        else:
            metrics = evaluate_score(sub[column].values, sub['patho'].values, direction='higher')
        metrics['condition'] = f'Per Locus LPS Length {column}'
        results.append(metrics)


def run_rexprt_block(other_pathogenic_ids, results):
    rexprt=pd.read_csv(os.path.join(ENRICH_DIR, 'TR_Explorer_v1.0.1.hg19.RExPRTscores.txt.gz'),sep='\t')
    rexprt=rexprt.rename(columns={'sampleID':'TRID'})
    rexprt['patho']=rexprt['TRID'].isin(other_pathogenic_ids).astype(int)
    for column in ['ensembleScore','ensembleMax']:
        sub = rexprt[[column, 'patho']].dropna(subset=[column])
        metrics = evaluate_score(sub[column].values, sub['patho'].values,
                                 direction='higher',
                                 fixed_thresholds={'binary': 0.5})
        metrics['condition'] = f'RExPRT {column}'
        results.append(metrics)


def run_distance_block(total_len, results):
    tad = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestTADBoundary.txt.gz'),
                      sep='\t', header=None, names=['TRID', 'TADDistance'])
    rad21 = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestRAD21Site.txt.gz'),
                        sep='\t', header=None, names=['TRID', 'RAD21Distance'])
    smc3 = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestSMC3Site.txt.gz'),
                       sep='\t', header=None, names=['TRID', 'SMC3Distance'])
    oreganno = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestORegAnnoFeature.txt.gz'),
                           sep='\t', header=None, names=['TRID', 'ORegAnnoDistance'])
    alu = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestAlu.txt.gz'),
                      sep='\t', header=None, names=['TRID', 'AluDistance'])
    estr = pd.read_csv(os.path.join(ENRICH_DIR, 'distanceToClosestESTR.txt.gz'),
                       sep='\t', header=None, names=['TRID', 'eSTRDistance'])

    maxVal=tad.loc[:,'TADDistance'].max()
    tad.loc[tad['TADDistance']==-1,'TADDistance']=maxVal
    maxVal=rad21.loc[:,'RAD21Distance'].max()
    rad21.loc[rad21['RAD21Distance']==-1,'RAD21Distance']=maxVal
    maxVal=smc3.loc[:,'SMC3Distance'].max()
    smc3.loc[smc3['SMC3Distance']==-1,'SMC3Distance']=maxVal
    maxVal=oreganno.loc[:,'ORegAnnoDistance'].max()
    oreganno.loc[oreganno['ORegAnnoDistance']==-1,'ORegAnnoDistance']=maxVal
    maxVal=alu.loc[:,'AluDistance'].max()
    alu.loc[alu['AluDistance']==-1,'AluDistance']=maxVal
    maxVal=estr.loc[:,'eSTRDistance'].max()
    estr.loc[estr['eSTRDistance']==-1,'eSTRDistance']=maxVal
   
    tad=tad.sort_values(by='TADDistance',ascending=True).drop_duplicates(subset='TRID',keep='first')
    rad21=rad21.sort_values(by='RAD21Distance',ascending=True).drop_duplicates(subset='TRID',keep='first')
    smc3=smc3.sort_values(by='SMC3Distance',ascending=True).drop_duplicates(subset='TRID',keep='first')
    oreganno=oreganno.sort_values(by='ORegAnnoDistance',ascending=True).drop_duplicates(subset='TRID',keep='first')
    alu=alu.sort_values(by='AluDistance',ascending=True).drop_duplicates(subset='TRID',keep='first')
    estr=estr.sort_values(by='eSTRDistance',ascending=True).drop_duplicates(subset='TRID',keep='first')

    combined = (total_len.merge(tad, how='inner', on='TRID')
                          .merge(rad21, how='inner', on='TRID')
                          .merge(smc3, how='inner', on='TRID')
                          .merge(oreganno, how='inner', on='TRID')
                          .merge(alu, how='inner', on='TRID')
                          .merge(estr, how='inner', on='TRID'))

    for column in ['TADDistance', 'RAD21Distance', 'SMC3Distance',
                   'ORegAnnoDistance', 'AluDistance', 'eSTRDistance']:
        sub = combined[[column, 'patho']].dropna(subset=[column])
        metrics = evaluate_score(sub[column].values, sub['patho'].values, direction='lower')
        metrics['condition'] = column
        results.append(metrics)


other_pathogenic_ids = load_pathogenic_ids()
results = []

total_len = run_allele_length_block(other_pathogenic_ids, results)
run_lps_block(other_pathogenic_ids, results)
run_rexprt_block(other_pathogenic_ids, results)
run_distance_block(total_len, results)

results_df = pd.DataFrame(results)
front = ['condition', 'direction', 'n_total', 'n_pathogenic', 'auroc']
ordered_cols = front + [c for c in results_df.columns if c not in front]
results_df = results_df[ordered_cols]

out_csv = os.path.join(ENRICH_DIR, 'results_collection_figure3B_classificationMetrics.csv')
results_df.to_csv('statsTables/SupplementaryFile3.csv', index=False)
# Note, some minor grooming and re-organizing of columns was done to transform this csv into the final SupplementaryFile3.xlsx file shared in the manuscript



# SF4: TR Constraint results

In [ ]:
import pandas as pd
training=pd.read_csv('assets/constraintAnalyses/training_10kbFromGenes_scoredNT3.bed.gz',sep='\t')
validation=pd.read_csv('assets/constraintAnalyses/validation_10kbFromGenes_scoredNT3.bed.gz',sep='\t')
testing=pd.read_csv('assets/constraintAnalyses/testing_within10kbOfGenes_scoredNT3.bed.gz',sep='\t')
training['class']='training'
validation['class']='validation'
testing['class']='testing'
combined=pd.concat([training.loc[:,['TRID','class','combinedLPSStdev','expectedCombinedLPSStdev','chr','start']],
                    validation.loc[:,['TRID','class','combinedLPSStdev','expectedCombinedLPSStdev','chr','start']],
                    testing.loc[:,['TRID','class','combinedLPSStdev','expectedCombinedLPSStdev','chr','start']]],ignore_index=True)
combined['OE']=(combined.loc[:,'combinedLPSStdev']+0.1)/(combined.loc[:,'expectedCombinedLPSStdev']+0.1)
combined=combined.sort_values(by=['chr','start'],ascending=True).loc[:,['TRID','class','combinedLPSStdev','expectedCombinedLPSStdev','OE']].reset_index(drop=True)
combined.to_csv('statsTables/SupplementaryFile4.txt.gz',sep='\t',index=False)

# PLVI Score

In [2]:
import pandas as pd
# load LPS Per Motif stats
lpsPerMotif=pd.read_csv('statsTables/DiscoveryCohort_lpsPerMotifStats.TR_Explorer_1.0.1.txt.gz',sep='\t')

# drop to the most common LPS motif per locus
lpsPerMotif=lpsPerMotif.sort_values(by=['TRID','N_motif'],ascending=False).drop_duplicates(subset='TRID',keep='first').reset_index(drop=True)

# function to convert motifs to their simplest rotational form
def simplest_motif(motif):
    i = (motif+motif).find(motif, 1, -1)
    if i != -1:
        motif = motif[:i]
    motifs = []
    b = len(motif)
    for i in range(b):
        c = motif[i:]+motif[:i]
        motifs.append(c)
    motifs = [x.upper() for x in motifs]  
    return(min(motifs))

# apply the simplest_motif function the data
lpsPerMotif['simpleMotif']=lpsPerMotif.loc[:,'LPS_Motif'].apply(simplest_motif)
lpsPerMotif['motifLen']=lpsPerMotif.loc[:,'simpleMotif'].str.len()

# load LPS Per Locus Stats
lps=pd.read_csv('statsTables/DiscoveryCohort_lpsPerLocusStats.TR_Explorer_1.0.1.txt.gz',sep='\t')

# add motifLen to lpsPerLocus dataframe
lps=lps.merge(lpsPerMotif.loc[:,['TRID','motifLen']],how='inner',on='TRID')
lps['PLVI']=lps.groupby('motifLen',observed=True)['std'].rank(pct=True)
lps.to_csv('statsTables/DiscoveryCohort_lpsPerLocusStats.TR_Explorer_1.0.1_withPLVI.txt.gz',sep='\t',index=False)


In [3]:
# repeat with Adotto catalog
import pandas as pd
# load LPS Per Motif stats
lpsPerMotif=pd.read_csv('statsTables/DiscoveryCohort_lpsPerMotifStats.Adotto.txt.gz',sep='\t')

# drop to the most common LPS motif per locus
lpsPerMotif=lpsPerMotif.sort_values(by=['TRID','N_motif'],ascending=False).drop_duplicates(subset='TRID',keep='first').reset_index(drop=True)

# function to convert motifs to their simplest rotational form
def simplest_motif(motif):
    i = (motif+motif).find(motif, 1, -1)
    if i != -1:
        motif = motif[:i]
    motifs = []
    b = len(motif)
    for i in range(b):
        c = motif[i:]+motif[:i]
        motifs.append(c)
    motifs = [x.upper() for x in motifs]  
    return(min(motifs))

# apply the simplest_motif function the data
lpsPerMotif['simpleMotif']=lpsPerMotif.loc[:,'LPS_Motif'].apply(simplest_motif)
lpsPerMotif['motifLen']=lpsPerMotif.loc[:,'simpleMotif'].str.len()

# load LPS Per Locus Stats
lps=pd.read_csv('statsTables/DiscoveryCohort_lpsPerLocusStats.Adotto.txt.gz',sep='\t')

# add motifLen to lpsPerLocus dataframe
lps=lps.merge(lpsPerMotif.loc[:,['TRID','motifLen']],how='inner',on='TRID')
lps['PLVI']=lps.groupby('motifLen',observed=True)['std'].rank(pct=True)
lps.to_csv('statsTables/DiscoveryCohort_lpsPerLocusStats.Adotto_withPLVI.txt.gz',sep='\t',index=False)
